# Import

In [8]:
import os
import shutil
import re
import pickle
from pathlib import Path


# ---------------- project & output roots ----------------
# Pick scratch if it exists, otherwise project
if Path("/vast/palmer/scratch/mcdougal/imc33/mod-extract").exists():
    PROJECT_ROOT = Path("/vast/palmer/scratch/mcdougal/imc33/mod-extract")
else:
    PROJECT_ROOT = Path("/gpfs/gibbs/project/mcdougal/imc33/mod-extract")

# Where the consolidated logs live
log_folder = PROJECT_ROOT / "logs"
run_log_path = log_folder / "run_log.txt"
success_dir = log_folder / "successful"

# Where the .mod sources live
mod_src_dir = PROJECT_ROOT / "data" / "raw" / "nmodl"

# Bundle failed outputs under the *script directory*/failed/<hash>/
# (This yields: code/failed/<hash>/<hash>.log and <hash>.mod)
SCRIPT_DIR = Path.cwd()
FAILED_BUNDLE_ROOT = SCRIPT_DIR / "failed"

# If space is tight, symlink instead of copying the .log/.mod into each bundle.
USE_SYMLINKS = False

# ---------------- utils ----------------
def ensure_dirs(*dirs):
    for d in dirs:
        Path(d).mkdir(parents=True, exist_ok=True)

def parse_run_log(log_path):
    success, fail = [], []
    with open(log_path, "r", encoding="utf-8", errors="ignore") as f:
        lines = f.readlines()
    for i, line in enumerate(lines):
        m = re.match(r"==== Processing (.+)\.mod ====", line.strip())
        if m:
            h = m.group(1)
            nxt = lines[i + 1].strip() if i + 1 < len(lines) else ""
            if "Successfully processed" in nxt:
                success.append(h)
            elif "failed" in nxt.lower():
                fail.append(h)
    return success, fail

def move_logs(hashes, src_dir, dst_dir):
    """For successful logs only; we still keep your success layout."""
    ensure_dirs(dst_dir)
    for h in hashes:
        src = Path(src_dir) / f"{h}.log"
        dst = Path(dst_dir) / f"{h}.log"
        if src.exists():
            shutil.copy2(src, dst)

def load_excluded_hashes(pipeline_dir):
    files = ["failed_excluded.pkl", "failed_neither.pkl"]
    excluded = set()
    for fname in files:
        fp = Path(pipeline_dir) / fname
        if fp.exists():
            with open(fp, "rb") as f:
                excluded.update(pickle.load(f))
    return excluded

# ---- reason-based skipping (e.g., SUFFIX not found) ----
SKIP_PATTERNS = [
    r"suffix\s+not\s+found",
    r"exclude_hashes",
]

def should_skip_failure_for_reason(log_text, patterns=SKIP_PATTERNS):
    t = log_text.lower()
    return any(re.search(p, t) for p in patterns)

def read_log_text(log_dir, hash_id):
    fp = Path(log_dir) / f"{hash_id}.log"
    if not fp.exists():
        return ""
    with open(fp, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()

def filter_failures_by_log_reason(fail_hashes, log_dir, extra_skip_patterns=None):
    pats = SKIP_PATTERNS[:] + (extra_skip_patterns or [])
    keep, skipped = [], []
    for h in fail_hashes:
        if should_skip_failure_for_reason(read_log_text(log_dir, h), pats):
            skipped.append(h)
        else:
            keep.append(h)
    return keep, skipped

# ---- NEW: bundle failed items under code/failed/<hash>/ ----
def _safe_link_or_copy(src: Path, dst: Path):
    if not src.exists():
        return False
    if dst.exists():
        return True  # already present
    if USE_SYMLINKS:
        try:
            dst.symlink_to(src)
        except FileExistsError:
            pass
    else:
        shutil.copy2(src, dst)
    return True

def bundle_failed_hash(hash_id, log_src_dir: Path, mod_src_dir: Path, bundle_root: Path):
    """
    Create: bundle_root/<hash>/
      - <hash>.log (from log_src_dir)
      - <hash>.mod (from mod_src_dir)
    Returns (has_log, has_mod)
    """
    bundle_dir = bundle_root / hash_id
    ensure_dirs(bundle_dir)

    log_src = Path(log_src_dir) / f"{hash_id}.log"
    mod_src = Path(mod_src_dir) / f"{hash_id}.mod"

    log_dst = bundle_dir / f"{hash_id}.log"
    mod_dst = bundle_dir / f"{hash_id}.mod"

    has_log = _safe_link_or_copy(log_src, log_dst)
    has_mod = _safe_link_or_copy(mod_src, mod_dst)

    missing = []
    if not has_log: missing.append("log")
    if not has_mod: missing.append("mod")
    if missing:
        print(f"⚠️  {hash_id}: missing {', '.join(missing)}")

    return has_log, has_mod

def bundle_all_failed(fail_hashes, log_src_dir: Path, mod_src_dir: Path, bundle_root: Path):
    ensure_dirs(bundle_root)
    bundled = 0
    for h in fail_hashes:
        has_log, has_mod = bundle_failed_hash(h, log_src_dir, mod_src_dir, bundle_root)
        if has_log or has_mod:
            bundled += 1
    return bundled

# ================ Main ================
ensure_dirs(success_dir, FAILED_BUNDLE_ROOT)

success_hashes, fail_hashes = parse_run_log(run_log_path)

# Remove explicitly excluded
pipeline_dir = PROJECT_ROOT / "data" / "pipeline"
excluded_hashes = load_excluded_hashes(pipeline_dir)
fail_hashes = [h for h in fail_hashes if h not in excluded_hashes]

# Remove failures due to SUFFIX-not-found / exclude_hashes
fail_hashes_filtered, skipped_reason_hashes = filter_failures_by_log_reason(
    fail_hashes, log_folder, extra_skip_patterns=None
)

# 1) Handle successes as before
move_logs(success_hashes, log_folder, success_dir)

# 2) NEW: Bundle fails: code/failed/<hash>/{<hash>.log,<hash>.mod}
bundled_count = bundle_all_failed(
    fail_hashes_filtered,
    log_src_dir=log_folder,
    mod_src_dir=mod_src_dir,
    bundle_root=FAILED_BUNDLE_ROOT,
)

print(f"✅ Moved {len(success_hashes)} successful logs → {success_dir}")
print(f"📦 Bundled {bundled_count} failed hashes → {FAILED_BUNDLE_ROOT}")
print(f"⏭️ Skipped {len(skipped_reason_hashes)} failures (SUFFIX not found / exclude_hashes)")

✅ Moved 426 successful logs → /vast/palmer/scratch/mcdougal/imc33/mod-extract/logs/successful
📦 Bundled 48 failed hashes → /vast/palmer/scratch/mcdougal/imc33/mod-extract/code/failed
⏭️ Skipped 68 failures (SUFFIX not found / exclude_hashes)


In [15]:
import os
import shutil
import re
import pickle
import pandas as pd
from pathlib import Path

# ---------------- project & output roots ----------------
# Pick scratch if it exists, otherwise project
if Path("/vast/palmer/scratch/mcdougal/imc33/mod-extract").exists():
    PROJECT_ROOT = Path("/vast/palmer/scratch/mcdougal/imc33/mod-extract")
else:
    PROJECT_ROOT = Path("/gpfs/gibbs/project/mcdougal/imc33/mod-extract")

# Where the consolidated logs live
log_folder = PROJECT_ROOT / "logs"
run_log_path = log_folder / "run_log.txt"
success_dir = log_folder / "successful"

# Where the .mod sources live
mod_src_dir = PROJECT_ROOT / "data" / "raw" / "nmodl"

# Bundle failed outputs under the *script directory*/failed/<rowid_type_subtype>/
# (This yields: code/failed/<rowid_type_subtype>/<hash>.log and <hash>.mod, plus hash.txt)
SCRIPT_DIR = Path.cwd()
FAILED_BUNDLE_ROOT = SCRIPT_DIR / "failed"

# If space is tight, symlink instead of copying the .log/.mod into each bundle.
USE_SYMLINKS = False

# ---------------- utils ----------------
def ensure_dirs(*dirs):
    for d in dirs:
        Path(d).mkdir(parents=True, exist_ok=True)

def parse_run_log(log_path):
    success, fail = [], []
    with open(log_path, "r", encoding="utf-8", errors="ignore") as f:
        lines = f.readlines()
    for i, line in enumerate(lines):
        m = re.match(r"==== Processing (.+)\.mod ====", line.strip())
        if m:
            h = m.group(1)
            nxt = lines[i + 1].strip() if i + 1 < len(lines) else ""
            if "Successfully processed" in nxt:
                success.append(h)
            elif "failed" in nxt.lower():
                fail.append(h)
    return success, fail

def move_logs(hashes, src_dir, dst_dir):
    """For successful logs only; we still keep your success layout."""
    ensure_dirs(dst_dir)
    for h in hashes:
        src = Path(src_dir) / f"{h}.log"
        dst = Path(dst_dir) / f"{h}.log"
        if src.exists():
            shutil.copy2(src, dst)

def load_excluded_hashes(pipeline_dir):
    files = ["failed_excluded.pkl", "failed_neither.pkl"]
    excluded = set()
    for fname in files:
        fp = Path(pipeline_dir) / fname
        if fp.exists():
            with open(fp, "rb") as f:
                excluded.update(pickle.load(f))
    return excluded

# ---- reason-based skipping (e.g., SUFFIX not found) ----
SKIP_PATTERNS = [
    r"suffix\s+not\s+found",
    r"exclude_hashes",
]

def should_skip_failure_for_reason(log_text, patterns=SKIP_PATTERNS):
    t = log_text.lower()
    return any(re.search(p, t) for p in patterns)

def read_log_text(log_dir, hash_id):
    fp = Path(log_dir) / f"{hash_id}.log"
    if not fp.exists():
        return ""
    with open(fp, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()

def filter_failures_by_log_reason(fail_hashes, log_dir, extra_skip_patterns=None):
    pats = SKIP_PATTERNS[:] + (extra_skip_patterns or [])
    keep, skipped = [], []
    for h in fail_hashes:
        if should_skip_failure_for_reason(read_log_text(log_dir, h), pats):
            skipped.append(h)
        else:
            keep.append(h)
    return keep, skipped

# ---------------- crosswalk: hash -> folder name ----------------
def _sanitize_segment(s: str) -> str:
    """
    Make a safe path segment: keep alnum, dash, underscore; convert spaces to underscores.
    Lowercase for consistency; strip leading/trailing underscores/dashes.
    """
    if s is None:
        s = ""
    s = str(s)
    s = s.replace(os.sep, "_").replace(" ", "_")
    s = re.sub(r"[^A-Za-z0-9_\-]+", "_", s)
    s = re.sub(r"_+", "_", s)
    return s.strip("_-").lower()

def load_hash_to_name_map(project_root: Path) -> dict:
    """
    Builds a mapping {file_hash: 'rowid_type_subtype'} using model_db_annotations.xlsx
    Prefers PROJECT_ROOT/data/raw/model_db_annotations.xlsx, but also tries ../data/raw/.
    Uses only rows where annotated == 'y'.
    """
    candidates = [
        project_root / "data" / "raw" / "model_db_annotations.xlsx",
        SCRIPT_DIR.parent / "data" / "raw" / "model_db_annotations.xlsx",
        SCRIPT_DIR / "data" / "raw" / "model_db_annotations.xlsx",
    ]
    fp = next((p for p in candidates if p.exists()), None)
    if fp is None:
        print("⚠️  Crosswalk not found; will fall back to using hash as folder name.")
        return {}

    df = pd.read_excel(fp, usecols=["row_id", "file_hash", "type", "subtype_1", "annotated"])
    df = df[df["annotated"].astype(str).str.lower().eq("y")]

    mapping = {}
    # Track used names to avoid collisions (append short hash if needed)
    used = set()
    for row in df.itertuples(index=False):
        row_id = _sanitize_segment(getattr(row, "row_id"))
        typ = _sanitize_segment(getattr(row, "type"))
        sub1 = _sanitize_segment(getattr(row, "subtype_1"))
        h = str(getattr(row, "file_hash"))

        base = "_".join([p for p in [row_id, typ, sub1] if p])
        if not base:
            base = "unnamed"

        name = base
        # Avoid directory name collisions across different hashes
        if name in used:
            # Append short hash tail for uniqueness
            short = h[:8] if h else "hash"
            name = f"{base}_{short}"
        used.add(name)
        mapping[h] = name
    return mapping

# ---- bundle failed items under code/failed/<rowid_type_subtype>/ ----
def _safe_link_or_copy(src: Path, dst: Path):
    if not src.exists():
        return False
    if dst.exists():
        return True  # already present
    if USE_SYMLINKS:
        try:
            dst.symlink_to(src)
        except FileExistsError:
            pass
    else:
        shutil.copy2(src, dst)
    return True

def bundle_failed_hash(hash_id, log_src_dir: Path, mod_src_dir: Path, bundle_root: Path, hash_to_name_map: dict):
    """
    Create: bundle_root/<rowid_type_subtype or hash>/
      - <hash>.log (from log_src_dir)
      - <hash>.mod (from mod_src_dir)
    Returns (has_log, has_mod)
    """
    folder_name = hash_to_name_map.get(hash_id, hash_id)
    bundle_dir = bundle_root / folder_name
    ensure_dirs(bundle_dir)

    log_src = Path(log_src_dir) / f"{hash_id}.log"
    mod_src = Path(mod_src_dir) / f"{hash_id}.mod"

    log_dst = bundle_dir / f"{hash_id}.log"
    mod_dst = bundle_dir / f"{hash_id}.mod"

    has_log = _safe_link_or_copy(log_src, log_dst)
    has_mod = _safe_link_or_copy(mod_src, mod_dst)

    missing = []
    if not has_log: missing.append("log")
    if not has_mod: missing.append("mod")
    if missing:
        print(f"⚠️  {hash_id} → {folder_name}: missing {', '.join(missing)}")

    return has_log, has_mod


def bundle_all_failed(fail_hashes, log_src_dir: Path, mod_src_dir: Path, bundle_root: Path, hash_to_name_map: dict):
    ensure_dirs(bundle_root)
    bundled = 0
    for h in fail_hashes:
        has_log, has_mod = bundle_failed_hash(h, log_src_dir, mod_src_dir, bundle_root, hash_to_name_map)
        if has_log or has_mod:
            bundled += 1
    return bundled

# ================ Main ================
ensure_dirs(success_dir, FAILED_BUNDLE_ROOT)

# 0) Crosswalk mapping (hash -> "rowid_type_subtype")
hash_to_name = load_hash_to_name_map(PROJECT_ROOT)

# 1) Parse run log
success_hashes, fail_hashes = parse_run_log(run_log_path)

# 2) Remove explicitly excluded
pipeline_dir = PROJECT_ROOT / "data" / "pipeline"
excluded_hashes = load_excluded_hashes(pipeline_dir)
fail_hashes = [h for h in fail_hashes if h not in excluded_hashes]

# 3) Remove failures due to SUFFIX-not-found / exclude_hashes
fail_hashes_filtered, skipped_reason_hashes = filter_failures_by_log_reason(
    fail_hashes, log_folder, extra_skip_patterns=None
)

# 4) Handle successes as before
move_logs(success_hashes, log_folder, success_dir)

# 5) Bundle fails: code/failed/<rowid_type_subtype>/{<hash>.log,<hash>.mod,hash.txt}
bundled_count = bundle_all_failed(
    fail_hashes_filtered,
    log_src_dir=log_folder,
    mod_src_dir=mod_src_dir,
    bundle_root=FAILED_BUNDLE_ROOT,
    hash_to_name_map=hash_to_name
)

print(f"✅ Moved {len(success_hashes)} successful logs → {success_dir}")
print(f"📦 Bundled {bundled_count} failed hashes → {FAILED_BUNDLE_ROOT}")
print(f"⏭️ Skipped {len(skipped_reason_hashes)} failures (SUFFIX not found / exclude_hashes)")


/home/imc33/.conda/envs/nn/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)
/home/imc33/.conda/envs/nn/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


✅ Moved 426 successful logs → /vast/palmer/scratch/mcdougal/imc33/mod-extract/logs/successful
📦 Bundled 48 failed hashes → /vast/palmer/scratch/mcdougal/imc33/mod-extract/code/failed
⏭️ Skipped 68 failures (SUFFIX not found / exclude_hashes)


In [9]:
!git add .
!git commit -m "restructured directories"
!git push

[main 09fc7cb] restructured directories
 97 files changed, 5654 insertions(+), 13 deletions(-)
 create mode 100644 code/failed/11cbdd1f59a9771461f9f08d7c8d2c09ddb7baf1716c09ab354c5f89b06cbcee/11cbdd1f59a9771461f9f08d7c8d2c09ddb7baf1716c09ab354c5f89b06cbcee.log
 create mode 100644 code/failed/11cbdd1f59a9771461f9f08d7c8d2c09ddb7baf1716c09ab354c5f89b06cbcee/11cbdd1f59a9771461f9f08d7c8d2c09ddb7baf1716c09ab354c5f89b06cbcee.mod
 create mode 100644 code/failed/1e47dfd343d0bed6630ec508e86d32fe62dae51a242b971568b166bc8056bf2e/1e47dfd343d0bed6630ec508e86d32fe62dae51a242b971568b166bc8056bf2e.log
 create mode 100644 code/failed/1e47dfd343d0bed6630ec508e86d32fe62dae51a242b971568b166bc8056bf2e/1e47dfd343d0bed6630ec508e86d32fe62dae51a242b971568b166bc8056bf2e.mod
 create mode 100644 code/failed/1e4e63055fe66aacf8bf889828d3498d31caf8c3cb70a78084927332ea1cb685/1e4e63055fe66aacf8bf889828d3498d31caf8c3cb70a78084927332ea1cb685.log
 create mode 100644 code/failed/1e4e63055fe66aacf8bf889828d3498d31caf8c3cb7